In [2]:
import importlib as il
import numpy as np
import more_itertools as mit
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import scipy.linalg as spl
import linetimer as lt

import gurobipy as gp

import gurobi_utils as gu
import dikin_utils as du
import plot_utils as pu

import example_loader as el
import miplib_loader as ml
import jsplib_loader as jl
import knapsack_loader as kl

status_lookup = {getattr(gp.GRB.Status, k): k for k in gp.GRB.Status.__dir__() if "A" <= k[0] <= "Z"}

%matplotlib inline

In [6]:
def get_A_b_c_l_u(mdl: gp.Model):
    mdl.update()
    A = mdl.getA()
    b = np.array(mdl.getAttr("RHS")).reshape(-1, 1)
    c = np.array(mdl.getAttr("Obj"))
    l = np.array(mdl.getAttr("LB"))
    u = np.array(mdl.getAttr("UB"))
    return A, b, c, l , u

def sparse_null_space(A, tol=1e-7):
    import sparseqr as rz
    if not sp.issparse(A):
        raise TypeError("Input must be a sparse matrix.")
    
    m, n = A.shape

    # Compute rank-revealing QR (with column permutation)
    Q, _, E, rank = rz.qr(A, economy=False)
    nullity = n - rank

    # Early return if full rank
    if nullity == 0:
        return sp.csc_matrix((n, 0))

    # Apply inverse permutation to align with original columns
    E_inv = np.argsort(E)  # Inverse permutation

    # Extract nullity trailing columns (explicitly)
    Z = Q.tocsr()[E_inv, -nullity:]
    return Z

def get_x0(mdl: gp.Model):
    cpy = mdl.copy()
    cpy.params.LogToConsole = 0
    gu.relax_int_or_bin_to_continuous(cpy)
    cpy.optimize()
    if cpy.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Relaxation not optimal: {status_lookup[cpy.status]}")
    x0 = np.array([v.X for v in cpy.getVars()]).reshape((-1, 1))
    return x0

In [ ]:
il.reload(kl)
il.reload(gu)

def adaptive_null_space(A):
    """
    Computes null space with tolerance automatically adjusted based on condition number
    """
    # Compute SVD
    u, s, vh = spl.svd(A, full_matrices=True)
    
    # Calculate condition number (handle case where min singular value is 0)
    max_s = s[0]
    min_s = s[-1] if s[-1] > 0 else s[s > 0][-1]
    cond = max_s / min_s
    
    # Machine epsilon for float64
    eps = np.finfo(np.float64).eps
    
    # Adaptive tolerance
    tol = max(A.shape) * cond * eps
    print(f"Adaptive tolerance: {tol:.2e}, Condition number: {cond:.2e}")
    
    # Find null space basis vectors
    rank = (s >= tol).sum()
    null_space = vh[rank:].conj().T
    
    return null_space

instances = kl.generate(3, 2, 20, 1, 60, 400, equality=True, seed=43)
for model in instances:
    # assumptions on the model: all equality constraints, fully linear objective & constraints, all vars >= 0, maximizing
    x0 = get_x0(model)
    A = model.getA()
    b = np.array(model.getAttr("RHS")).reshape((-1, 1))
    m, n = A.shape
    assert n > m and np.allclose(A @ x0, b)
    # N = spl.null_space(A.todense(), check_finite=False, rcond=1e-7, lapack_driver='gesvd') # sparse_null_space(A)
    N = adaptive_null_space(A.todense())
    c = np.array(model.getAttr("Obj")).reshape((-1, 1))
    ub = np.array(model.getAttr("UB")).reshape((-1, 1)) # can't go without this?

    mdl2 = gp.Model(model.ModelName + "_tfrm")
    mdl2.params.IntegralityFocus = 1
    mdl2.params.IntFeasTol = 1e-8
    z = mdl2.addMVar((n - m, 1), lb=-gp.GRB.INFINITY, vtype='C', name='z')
    print(c.shape, x0.shape, A.shape, N.shape, z.shape)
    y = mdl2.addMVar((n, 1), ub=ub, vtype='I', name='y')
    # mdl2.setObjective(c.T @ x0 + c.T @ N @ z, gp.GRB.MAXIMIZE)
    mdl2.setObjective(c.T @ y, gp.GRB.MAXIMIZE)
    # mdl2.addConstr(N @ z >= -x0)
    mdl2.addConstr(x0 + N@z == y)
    mdl2.params.LogToConsole = 0
    with lt.CodeTimer("Transformed optimization time"):
        mdl2.optimize()
    if mdl2.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Transformation not optimal: {status_lookup[mdl2.status]}")
    x1 = y.X
    
    model.params.LogToConsole = 0
    with lt.CodeTimer("Original optimization time"):
        model.optimize()
    if model.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Original model not optimal: {status_lookup[model.status]}")
    x2 = np.array([v.X for v in model.getVars()]).reshape((-1, 1))
    
    print(f"Objective value: {mdl2.ObjVal}. Expected: {model.ObjVal}")
    print(f"Transformed solution: {x1.T}")
    print(f"Original solution: {x2.T}")
    

    


Set parameter LogToConsole to value 0


   Relaxed 20 variables on knapsack_2_20_0_copy
Adaptive tolerance: 1.56e-14, Condition number: 3.52e+00
Set parameter IntegralityFocus to value 1
Set parameter IntFeasTol to value 1e-08
(20, 1) (20, 1) (2, 20) (20, 18) (18, 1)
Set parameter LogToConsole to value 0
Code block 'Transformed optimization time' took: 2807.84753 ms
Set parameter LogToConsole to value 0
Code block 'Original optimization time' took: 3453.23283 ms
Objective value: 90760.0. Expected: 90760.0
Transformed solution: [[ 0.  0. 52.  0. 47.  0. 53. 13. 24.  1. 29.  0. 24.  0.  6. 43. 24. 37.
   0. 15.]]
Original solution: [[ 0.  0. 52.  0. 47.  0. 53. 13. 24.  1. 29.  0. 24.  0.  6. 43. 24. 37.
   0. 15.]]
Set parameter LogToConsole to value 0
   Relaxed 20 variables on knapsack_2_20_1_copy
Adaptive tolerance: 1.33e-14, Condition number: 3.00e+00
Set parameter IntegralityFocus to value 1
Set parameter IntFeasTol to value 1e-08
(20, 1) (20, 1) (2, 20) (20, 18) (18, 1)
Set parameter LogToConsole to value 0
Code block '

In [4]:
import ntl_wrapper as ntl
il.reload(ntl)
def test_paper():
    # from SOLVING A SYSTEM OF LINEAR DIOPHANTINE EQUATIONS WITH LOWER AND UPPER BOUNDS ON THE VARIABLES
    A = np.array([[6, 1, 3, 3, 0, 0], [0, 0, 0, 0, 2, 1], [0, 0, 4, 1, 0, 2]], dtype=np.int64)
    m, n = A.shape
    b = np.array([[17], [11], [27]], dtype=np.int64)
    N1 = max(np.linalg.norm(b, np.inf).item(), np.linalg.norm(A, np.inf).item()) * 4
    N2 = N1 * 4
    B = np.block([[np.eye(n, dtype=np.int64), np.zeros((n, 1), dtype=np.int64)],
                        [np.zeros((1, n), dtype=np.int64), np.array([N1])],
                        [N2 * A, -N2 * b]]).astype(np.int64, order='C')

    B_red = B.copy()
    rank, det, U = ntl.lll(B_red, 75, 100)
    # B @ U == B_red
    print(B_red)

    U = du.seysen_integer_matrix(B_red, 1)
    print(B_red)

test_paper()

[[  0   1   1   0   0   0   1]
 [ -3  -3   0  -1   1   0   0]
 [  1  -1   2   5   0  -1   0]
 [  0   0  -4   1   0   1  -2]
 [  1  -1   1   4   0  -1   0]
 [ -2   2  -2   3   0   2   1]
 [  0   0   0 108   0   0   0]
 [  0   0   0   0 432   0   0]
 [  0   0   0   0   0   0 432]
 [  0   0   0   0   0 432   0]]
[[  0   1   1   0   0   0   1]
 [ -3  -3   0  -1   1   0   0]
 [  1  -1   2   5   0  -1   0]
 [  0   0  -4   1   0   1  -2]
 [  1  -1   1   4   0  -1   0]
 [ -2   2  -2   3   0   2   1]
 [  0   0   0 108   0   0   0]
 [  0   0   0   0 432   0   0]
 [  0   0   0   0   0   0 432]
 [  0   0   0   0   0 432   0]]


In [15]:
import hsnf
il.reload(du)
import ntl_wrapper as ntl
il.reload(ntl)

def solve_via_snf(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    D, L, R = hsnf.smith_normal_form(A)
    m, n = A.shape
    rank = np.count_nonzero(np.diag(D))  # Number of non-zero diagonal entries in D
    
    # Step 1: Compute L b
    Lb = L @ b
    
    # Step 2: Solve D y = L b
    y_p = np.zeros((n, 1), dtype=int)  # Particular solution for y
    
    for i in range(rank):
        d_i = D[i, i]
        if d_i == 0:
            if Lb[i, 0] != 0:
                raise ValueError("No solution exists (inconsistent system)")
        else:
            if Lb[i, 0] % d_i != 0:
                raise ValueError("No integer solution exists (Lb not divisible by D[i,i])")
            y_p[i, 0] = Lb[i, 0] // d_i
    
    # Step 3: Compute x_p = R y_p
    x_p = R @ y_p

    assert np.allclose(A @ x_p, b), "x_p is not a solution to Ax = b"
    null_basis = R[:, rank:]
    # assert np.allclose(A @ N, 0), "invalid nulspace basis"

    return x_p, null_basis

def solve_via_LLL(A: np.ndarray, b: np.ndarray, check_gcd=False) -> np.ndarray:
    m, n = A.shape
    A = A.astype(np.int64)
    b = b.astype(np.int64)

    if check_gcd:
        for i in range(m):
            # find GCD of the row
            gcd = np.gcd.reduce(A[i, :], axis=1).item()
            if gcd > 1:
                print(f"Row GCD: {gcd}")
                if b[i, 0].item() % gcd != 0:
                    raise ValueError("No integer solution exists (b not divisible by GCD)")
                # divide the row by the GCD
                A[i, :] //= gcd
                b[i, 0] //= gcd

    N1 = max(np.linalg.norm(b, np.inf).item(), np.linalg.norm(A, np.inf).item()) * 2
    N2 = N1 * 2
    B = np.block([[np.eye(n, dtype=np.int64), np.zeros((n, 1), dtype=np.int64)],
                        [np.zeros((1, n), dtype=np.int64), np.array([N1])],
                        [N2 * A, -N2 * b]]).astype(np.int64, order='C')
    # B = sp.block_array([[sp.eye(n), sp.csr_array((n, 1))],
    #                     [sp.csr_array((1, n)), N1],
    #                     [N2 * A, -N2 * b]])
    B_red = B.copy()
    rank, det, U = ntl.lll(B_red, 3, 4)
    assert B_red[n, n-m].item() == N1
    x_p = B_red[0:n, n-m]
    assert np.allclose(A @ x_p, b)
    null_space = B_red[0:n, 0:n-m]

    return x_p, null_space

instances = kl.generate(3, 2, 30, 5, 10, 1000, equality=True, seed=42)
for model in instances:
    model.params.LogToConsole = 1
    # assumptions on the model: all equality constraints, fully linear objective & constraints, all vars >= 0, maximizing
    A = model.getA().todense()
    b = np.array(model.getAttr("RHS")).reshape((-1, 1))
    c = np.array(model.getAttr("Obj")).reshape((-1, 1))
    ub = np.array(model.getAttr("UB")).reshape((-1, 1))

    with lt.CodeTimer("Original optimization time"):
        model.optimize()
    if model.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Original model not optimal: {status_lookup[model.status]}")
    print(f"Original objective value: {model.ObjVal}")

    xp, N = solve_via_LLL(A, b)
    print(f"Number of negative components: {np.sum(xp < 0)}")
    # xp, N = solve_via_snf(A, b)
    # now I have an integer null space and an integer starting solution (that may violate bounds)

    mdl2 = gp.Model(model.ModelName + "_tfrm")
    z = mdl2.addMVar((N.shape[1], 1), lb=-gp.GRB.INFINITY, vtype='I', name='z')
    mdl2.setObjective(c.T @ xp + c.T @ N @ z, gp.GRB.MAXIMIZE)
    mdl2.addConstr(xp + N@z >= 0)
    mdl2.addConstr(xp + N@z <= ub)
    # mdl2.params.NumericFocus = 3
    # mdl2.params.DualReductions = 0
    mdl2.params.LogToConsole = 1
    with lt.CodeTimer("Transformed optimization time"):
        mdl2.optimize()
    if mdl2.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Transformation not optimal: {status_lookup[mdl2.status]}")
    print("Objective value: ", mdl2.ObjVal)
    mdl2.dispose()
    print()
    break


Set parameter LogToConsole to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (linux64 - "EndeavourOS")

CPU model: Intel(R) Core(TM) i7-8750H CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Academic license 2586148 - for non-commercial use only - registered to br___@vt.edu
Optimize a model with 2 rows, 30 columns and 60 nonzeros
Model fingerprint: 0x6ef058af
Variable types: 0 continuous, 30 integer (0 binary)
Coefficient statistics:
  Matrix range     [2e+01, 1e+03]
  Objective range  [1e+01, 1e+03]
  Bounds range     [5e+00, 1e+01]
  RHS range        [5e+04, 6e+04]
Presolve time: 0.00s
Presolved: 2 rows, 30 columns, 58 nonzeros
Variable types: 0 continuous, 30 integer (0 binary)

Root relaxation: objective 8.774003e+04, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap 

In [2]:
import ntl_wrapper as ntl

def solve_ineq_via_LLL(A: np.ndarray, b: np.ndarray, check_gcd=False) -> np.ndarray:
    A = A.astype(np.int64)
    b = b.astype(np.int64)
    m, n = A.shape

    N1 = max(np.linalg.norm(b, np.inf).item(), np.linalg.norm(A, np.inf).item()) * 5
    N2 = N1 * 5
    B = np.block([[np.eye(n + m, dtype=np.int64), np.zeros((n + m, 1), dtype=np.int64)],
                        [np.zeros((1, n + m), dtype=np.int64), np.array([N1])],
                        [N2 * A, N2 * np.eye(m), -N2 * b]]).astype(np.int64, order='C')
    # B = sp.block_array([[sp.eye(n), sp.csr_array((n, 1))],
    #                     [sp.csr_array((1, n)), N1],
    #                     [N2 * A, -N2 * b]])
    B_red = B.copy()
    B_spc = sp.csr_matrix(B_red)
    entries = B_spc.getnnz()
    print(f"Number of entries in B: {entries} on {B_red.shape}")
    rank, det, U = ntl.lll(B_red, 75, 100)
    assert B_red[n+m, n].item() == N1
    x_p = B_red[0:n, n:n+1]
    s_p = B_red[n:n+m, n:n+1]
    assert np.allclose(A @ x_p + s_p, b)
    x_null = B_red[0:n, 0:n]
    s_null = B_red[n:n+m, 0:n]

    return x_p, s_p, x_null, s_null

instances = jl.get_instances()
for instance in list(instances.values())[7:8]:
    model = instance.as_gurobi_balas_model(True, False)
    model.params.LogToConsole = 0
    gu.standardize_gt_to_lt(model)
    assert not any(c.Sense == gp.GRB.EQUAL for c in model.getConstrs()), "Model has equality constraints, expected all inequalities"
    # assumptions on the model: all equality constraints, fully linear objective & constraints, all vars >= 0, maximizing
    A = model.getA().todense()
    b = np.array(model.getAttr("RHS")).reshape((-1, 1))
    c = np.array(model.getAttr("Obj")).reshape((-1, 1))
    ub = np.array(model.getAttr("UB")).reshape((-1, 1))
    for v in model.getVars():
        if v.VType == gp.GRB.BINARY:
            ub[v.index] = 1

    with lt.CodeTimer("Original optimization time"):
        model.optimize()
    if model.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Original model not optimal: {status_lookup[model.status]}")
    print(f"Original objective value: {model.ObjVal}")

    xp, sp, xN, sN = solve_ineq_via_LLL(A, b)
    # left off: try the FP version of the transformation
    print(f"Number of negative components: {np.sum(xp < 0)} / {len(xp)}")
    print(f"Number of negative slacks: {np.sum(sp < 0)} / {len(sp)}")

    mdl2 = gp.Model(model.ModelName + "_tfrm")
    z = mdl2.addMVar((xN.shape[1], 1), lb=-gp.GRB.INFINITY, vtype='I', name='z')
    mdl2.setObjective((c.T @ xp).item() + c.T @ xN @ z, model.ModelSense)
    mdl2.addConstr(xp + xN@z >= 0)
    mdl2.addConstr(sp + sN@z >= 0)
    mdl2.addConstr(xp + xN@z <= ub)
    # mdl2.params.NumericFocus = 3
    # mdl2.params.DualReductions = 0
    mdl2.params.LogToConsole = 0
    with lt.CodeTimer("Transformed optimization time"):
        mdl2.optimize()
    if mdl2.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Transformation not optimal: {status_lookup[mdl2.status]}")
    print("Objective value: ", mdl2.ObjVal)
    mdl2.dispose()
    print()


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2586148
Academic license 2586148 - for non-commercial use only - registered to br___@vt.edu


Set parameter LogToConsole to value 0
   Negated 1000 constraints on abz5
Code block 'Original optimization time' took: 4839.39901 ms
Original objective value: 1234.0
Number of entries in B: 7002 on (3102, 2102)


KeyboardInterrupt: 